In [1]:
import os
import pandas as pd
from hashlib import md5
from PIL import Image

## Baseline clustering

In [4]:
baseline_data_path = "/Users/lukasb/Documents/data/surfaceClassification/baseline"
artifacts_path = "./artifacts/baseline"

In [5]:
baseline_images = []

extensions = [".png", ".jpg", ".jpeg"]

for dirpath, dirnames, filenames in os.walk(baseline_data_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            img = Image.open(file_path)
            file_path = file_path.removeprefix(baseline_data_path)
            image_id = md5(img.tobytes()).hexdigest()
            baseline_images.append({"image_id": image_id, "file_path": file_path, "label": label})


df = pd.DataFrame(baseline_images)

In [6]:
print(f"Number of images in baseline: {len(baseline_images)}")

Number of images in baseline: 5586


In [7]:
duplicates_ids = df[df.duplicated(subset=['image_id'], keep="first")]['image_id'].unique()

print(f"Number of duplicate images in baseline dataset: {len(duplicates_ids)}")

# Mark duplicates in the dataframe
df['is_duplicate'] = df['image_id'].isin(duplicates_ids)

Number of duplicate images in baseline dataset: 44


In [8]:
df.to_csv(os.path.join(artifacts_path, "baseline_images.csv"), index=False)

In [9]:
df_no_duplicates = df.drop_duplicates(subset=["image_id"], keep="first").copy()
print(f"Rows without duplicates: {len(df_no_duplicates)}")

df_no_duplicates.to_csv(os.path.join(artifacts_path, "baseline_images_no_duplicates.csv"), index=False)

Rows without duplicates: 5542


## Find near duplicates with pretrained model

In [9]:
import torch
import open_clip
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else device

# Load CLIP model and preprocess
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model.to(device).eval()

@torch.no_grad()
def get_clip_embedding(img):
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    feats = model.encode_image(img_tensor)
    return torch.nn.functional.normalize(feats, dim=-1)

embeddings = []
for row in tqdm(df_no_duplicates.itertuples(index=False), total=len(df_no_duplicates)):
    file_path = baseline_data_path + row.file_path
    img = Image.open(file_path).convert("RGB")
    embedding = get_clip_embedding(img).cpu().numpy().flatten()
    embeddings.append({"image_id": row.image_id, "embedding": embedding})

# Save embeddings
embeddings_array = np.array([e["embedding"] for e in embeddings])
image_ids = [e["image_id"] for e in embeddings]

np.save(os.path.join(artifacts_path, "image_embeddings.npy"), embeddings_array)
pd.DataFrame({"image_id": image_ids}).to_csv(os.path.join(artifacts_path, "image_embeddings_ids.csv"), index=False)

print(f"Generated embeddings for {len(embeddings)} images")
print(f"Embedding shape: {embeddings_array.shape}")


/home/lukasb/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/lukasb/miniforge3/envs/defect-detection/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
100%|██████████| 5542/5542 [00:28<00:00, 192.47it/s]


Generated embeddings for 5542 images
Embedding shape: (5542, 512)


In [10]:
embeddings_dict = {image_id: embeddings_array[idx] for idx, image_id in enumerate(image_ids)}

import faiss

dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_array)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}")

# search for 20 nearest neighbors
k = 20
D, I = index.search(embeddings_array, k)
D, I = torch.from_numpy(D), torch.from_numpy(I)#
# D: distances, I: indices of nearest neighbors

FAISS index built with 5542 vectors of dimension 512


In [11]:
high_threshold = 0.99
medium_threshold = 0.92
low_threshold = 0.85


# Calculate cosine similarity from L2 distances
# FAISS IndexFlatL2 returns squared L2 distances.
# For unit-normalized vectors: cos_sim = 1 - (squared_l2 / 2)
similarities = 1 - (D / 2)
similarities = torch.clamp(similarities, -1.0, 1.0)

# Create similarity pairs with image IDs
similarity_pairs = []
seen_pairs = set()
for i in range(len(similarities)):
    for j in range(1, k):  # Skip first neighbor (itself)
        img_id_1 = image_ids[i]
        img_id_2 = image_ids[I[i][j].item()]
        sim_score = similarities[i][j].item()
        pair_key = (img_id_1, img_id_2) if img_id_1 < img_id_2 else (img_id_2, img_id_1)
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        similarity_pairs.append({
            "image_id_1": img_id_1,
            "image_id_2": img_id_2,
            "similarity": sim_score
        })

similarity_df = pd.DataFrame(similarity_pairs)

# Save all similarity pairs to CSV
similarity_df.to_csv(os.path.join(artifacts_path, "similarity_pairs.csv"), index=False)
print(f"Saved {len(similarity_df)} similarity pairs to {os.path.join(artifacts_path, 'similarity_pairs.csv')}")

Saved 79615 similarity pairs to ./artifacts/baseline/similarity_pairs.csv
